In [1]:
import torch
import torch.nn as nn
from torch import Tensor
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader,SubsetRandomSampler,ConcatDataset
import torch.nn.functional as F

from model import SeqNN

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [4]:
model = SeqNN()
model = model.to(device)

In [5]:
from torchinfo import summary

summary(model, input_size=(2, 4, 1048576), col_names=["output_size", "num_params"])

PyTorch Running Mean: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
PyTorch Running Var: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1.], device='cuda:0')


Layer (type:depth-idx)                   Output Shape              Param #
SeqNN                                    [2, 96, 1048576]          741,733
├─StochasticReverseComplement: 1-1       [2, 4, 1048576]           --
├─StochasticShift: 1-2                   [2, 4, 1048576]           --
├─ReLU: 1-3                              [2, 4, 1048576]           --
├─ConvBlock: 1-4                         [2, 96, 1048576]          --
│    └─Conv1d: 2-1                       [2, 96, 1048576]          4,224
│    └─BatchNorm1d: 2-2                  [2, 96, 1048576]          192
Total params: 746,149
Trainable params: 746,149
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 8.86
Input size (MB): 33.55
Forward/backward pass size (MB): 3221.23
Params size (MB): 0.02
Estimated Total Size (MB): 3254.80

In [6]:
for name, module in model.named_modules():
    print(name, "->", module)

 -> SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (re_lu): ReLU()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 96, kernel_size=(11,), stride=(1,), padding=(5,), bias=False)
    (batch_norm): BatchNorm1d(96, eps=0.001, momentum=0.0735, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(96, 96, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(96, eps=0.001, momentum=0.0735, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(96, 96, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(96, eps=0.001, momentum=0.0735, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_

In [7]:
state_dict = model.state_dict()

In [8]:
for key in state_dict.keys():
    print(key)

conv_block_1.conv.weight
conv_block_1.batch_norm.weight
conv_block_1.batch_norm.bias
conv_block_1.batch_norm.running_mean
conv_block_1.batch_norm.running_var
conv_block_1.batch_norm.num_batches_tracked
conv_tower.conv_tower.1.weight
conv_tower.conv_tower.2.weight
conv_tower.conv_tower.2.bias
conv_tower.conv_tower.2.running_mean
conv_tower.conv_tower.2.running_var
conv_tower.conv_tower.2.num_batches_tracked
conv_tower.conv_tower.5.weight
conv_tower.conv_tower.6.weight
conv_tower.conv_tower.6.bias
conv_tower.conv_tower.6.running_mean
conv_tower.conv_tower.6.running_var
conv_tower.conv_tower.6.num_batches_tracked
conv_tower.conv_tower.9.weight
conv_tower.conv_tower.10.weight
conv_tower.conv_tower.10.bias
conv_tower.conv_tower.10.running_mean
conv_tower.conv_tower.10.running_var
conv_tower.conv_tower.10.num_batches_tracked
conv_tower.conv_tower.13.weight
conv_tower.conv_tower.14.weight
conv_tower.conv_tower.14.bias
conv_tower.conv_tower.14.running_mean
conv_tower.conv_tower.14.running_var


In [9]:
import h5py

In [10]:
h5_file = h5py.File("/project/fudenber_735/tensorflow_models/akita/v1/model_best.h5", "r")

In [11]:
print(list(h5_file["model_weights"].keys()))

['add', 'add_1', 'add_10', 'add_11', 'add_12', 'add_13', 'add_2', 'add_3', 'add_4', 'add_5', 'add_6', 'add_7', 'add_8', 'add_9', 'average_to2d', 'batch_normalization', 'batch_normalization_1', 'batch_normalization_10', 'batch_normalization_11', 'batch_normalization_12', 'batch_normalization_13', 'batch_normalization_14', 'batch_normalization_15', 'batch_normalization_16', 'batch_normalization_17', 'batch_normalization_18', 'batch_normalization_19', 'batch_normalization_2', 'batch_normalization_20', 'batch_normalization_21', 'batch_normalization_22', 'batch_normalization_23', 'batch_normalization_24', 'batch_normalization_25', 'batch_normalization_26', 'batch_normalization_27', 'batch_normalization_28', 'batch_normalization_29', 'batch_normalization_3', 'batch_normalization_30', 'batch_normalization_31', 'batch_normalization_32', 'batch_normalization_33', 'batch_normalization_34', 'batch_normalization_35', 'batch_normalization_36', 'batch_normalization_37', 'batch_normalization_38', 'ba

In [12]:
def assign_conv_weights(h5_file, tf_layer_path, pytorch_conv_layer):
    """
    Assign weights from a TensorFlow convolutional layer to a PyTorch convolutional layer.
    
    Parameters:
        h5_file (h5py.File): The HDF5 file containing TensorFlow weights.
        tf_layer_path (str): Path to the TensorFlow layer in the HDF5 file.
        pytorch_conv_layer (torch.nn.Conv1d or torch.nn.Conv2d): The PyTorch convolutional layer.

    """
    # Access TensorFlow kernel weights
    tf_weights = h5_file[tf_layer_path]['kernel:0'][:]
    
    # Convert to PyTorch tensor
    pytorch_weights = torch.tensor(tf_weights, dtype=torch.float32)
    
    # Transpose TensorFlow weights to match PyTorch's format
    pytorch_weights = pytorch_weights.permute(2, 1, 0)
    
    # Ensure the shapes match
    assert pytorch_weights.shape == pytorch_conv_layer.weight.data.shape, \
        f"Shape mismatch: {pytorch_weights.shape} vs {pytorch_conv_layer.weight.data.shape}"
    
    # Assign weights to the PyTorch layer
    pytorch_conv_layer.weight.data = pytorch_weights

    print(f"Assigned weights to PyTorch layer: {pytorch_conv_layer}")

In [13]:
def assign_batch_norm_weights(h5_file, tf_layer_path, pytorch_batch_norm_layer):
    """
    Assign batch normalization weights from TensorFlow to PyTorch.
    
    Parameters:
        h5_file (h5py.File): The HDF5 file containing TensorFlow weights.
        tf_layer_path (str): Path to the TensorFlow batch normalization layer in the HDF5 file.
        pytorch_batch_norm_layer (torch.nn.BatchNorm1d or torch.nn.BatchNorm2d): The PyTorch batch normalization layer.
    """
    # Access TensorFlow BatchNorm parameters
    batch_norm_group = h5_file[tf_layer_path]

    # Extract TensorFlow parameters
    beta = batch_norm_group["beta:0"][:]  # Bias (beta in TensorFlow)
    gamma = batch_norm_group["gamma:0"][:]  # Scale (gamma in TensorFlow)
    moving_mean = batch_norm_group["moving_mean:0"][:]  # Running mean
    moving_variance = batch_norm_group["moving_variance:0"][:]  # Running variance

    # Convert to PyTorch tensors
    beta_tensor = torch.tensor(beta, dtype=torch.float32)
    gamma_tensor = torch.tensor(gamma, dtype=torch.float32)
    moving_mean_tensor = torch.tensor(moving_mean, dtype=torch.float32)
    moving_variance_tensor = torch.tensor(moving_variance, dtype=torch.float32)

    # Assign values to the PyTorch BatchNorm layer
    pytorch_batch_norm_layer.bias.data = beta_tensor
    pytorch_batch_norm_layer.weight.data = gamma_tensor
    pytorch_batch_norm_layer.running_mean.data = moving_mean_tensor
    pytorch_batch_norm_layer.running_var.data = moving_variance_tensor

    print(f"Assigned batch normalization weights to PyTorch layer: {pytorch_batch_norm_layer}")


In [14]:
# ConvBlock

assign_conv_weights(h5_file, "model_weights/conv1d/conv1d", model.conv_block_1.conv)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization/batch_normalization", model.conv_block_1.batch_norm)

Assigned weights to PyTorch layer: Conv1d(4, 96, kernel_size=(11,), stride=(1,), padding=(5,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(96, eps=0.001, momentum=0.0735, affine=True, track_running_stats=True)


In [17]:
h5_file["model_weights/batch_normalization/batch_normalization"]["moving_mean:0"][:]

array([ 6.50698394e-02,  3.56163621e-01, -1.16524458e-01,  1.80145949e-01,
       -2.04334453e-01,  7.41125941e-01, -3.07366192e-01, -2.80924499e-01,
        4.47416067e-01, -5.32208622e-01,  3.89679819e-01,  1.07605457e-01,
        2.08657011e-01, -2.89328843e-01, -1.01516917e-01, -4.79835942e-02,
        2.88119674e-01, -7.79726058e-02,  1.73236206e-01, -7.86186308e-02,
        1.06632099e-01,  4.82826293e-01,  1.44133523e-01,  6.36939824e-01,
       -4.00668740e-01,  1.36201277e-01, -8.01831037e-02, -3.26570004e-01,
        2.99212694e-01,  1.28547519e-01, -1.06758460e-01,  4.10168827e-01,
        2.43843198e-01,  1.08542812e+00, -6.59783006e-01, -1.54871091e-01,
        7.14669168e-01,  9.71917659e-02, -1.23264425e-01, -1.43297315e-01,
       -4.03293908e-01, -1.92807019e-01,  4.24875319e-01,  2.56606519e-01,
        3.94721568e-01, -8.92990306e-02, -5.32825291e-01, -4.00303036e-01,
        9.04211760e-01,  5.24015844e-01, -1.09703355e-01,  2.87230134e-01,
        3.71010900e-01,  

In [ ]:
# ConvTower
# 1
assign_conv_weights(h5_file, "model_weights/conv1d_1/conv1d_1", model.conv_tower.conv_tower[1])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_1/batch_normalization_1", model.conv_tower.conv_tower[2])

In [ ]:
# 2
assign_conv_weights(h5_file, "model_weights/conv1d_2/conv1d_2", model.conv_tower.conv_tower[5])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_2/batch_normalization_2", model.conv_tower.conv_tower[6])

In [ ]:
# 3
assign_conv_weights(h5_file, "model_weights/conv1d_3/conv1d_3", model.conv_tower.conv_tower[9])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_3/batch_normalization_3", model.conv_tower.conv_tower[10])

In [ ]:
# 4
assign_conv_weights(h5_file, "model_weights/conv1d_4/conv1d_4", model.conv_tower.conv_tower[13])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_4/batch_normalization_4", model.conv_tower.conv_tower[14])

In [ ]:
# 5
assign_conv_weights(h5_file, "model_weights/conv1d_5/conv1d_5", model.conv_tower.conv_tower[17])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_5/batch_normalization_5", model.conv_tower.conv_tower[18])

In [ ]:
# 6
assign_conv_weights(h5_file, "model_weights/conv1d_6/conv1d_6", model.conv_tower.conv_tower[21])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_6/batch_normalization_6", model.conv_tower.conv_tower[22])

In [ ]:
# 7
assign_conv_weights(h5_file, "model_weights/conv1d_7/conv1d_7", model.conv_tower.conv_tower[25])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_7/batch_normalization_7", model.conv_tower.conv_tower[26])

In [ ]:
# 8
assign_conv_weights(h5_file, "model_weights/conv1d_8/conv1d_8", model.conv_tower.conv_tower[29])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_8/batch_normalization_8", model.conv_tower.conv_tower[30])

In [ ]:
# 9
assign_conv_weights(h5_file, "model_weights/conv1d_9/conv1d_9", model.conv_tower.conv_tower[33])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_9/batch_normalization_9", model.conv_tower.conv_tower[34])

In [ ]:
# 10
assign_conv_weights(h5_file, "model_weights/conv1d_10/conv1d_10", model.conv_tower.conv_tower[37])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_10/batch_normalization_10", model.conv_tower.conv_tower[38])

In [ ]:
# ResidualDilatedBlock

# 1
assign_conv_weights(h5_file, "model_weights/conv1d_11/conv1d_11", model.residual1d_block1.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_11/batch_normalization_11", model.residual1d_block1.bn1)

assign_conv_weights(h5_file, "model_weights/conv1d_12/conv1d_12", model.residual1d_block1.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_12/batch_normalization_12", model.residual1d_block1.bn2)

In [ ]:
# 2
assign_conv_weights(h5_file, "model_weights/conv1d_13/conv1d_13", model.residual1d_block2.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_13/batch_normalization_13", model.residual1d_block2.bn1)

assign_conv_weights(h5_file, "model_weights/conv1d_14/conv1d_14", model.residual1d_block2.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_14/batch_normalization_14", model.residual1d_block2.bn2)

In [ ]:
# 3
assign_conv_weights(h5_file, "model_weights/conv1d_15/conv1d_15", model.residual1d_block3.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_15/batch_normalization_15", model.residual1d_block3.bn1)

assign_conv_weights(h5_file, "model_weights/conv1d_16/conv1d_16", model.residual1d_block3.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_16/batch_normalization_16", model.residual1d_block3.bn2)

In [ ]:
# 4
assign_conv_weights(h5_file, "model_weights/conv1d_17/conv1d_17", model.residual1d_block4.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_17/batch_normalization_17", model.residual1d_block4.bn1)

assign_conv_weights(h5_file, "model_weights/conv1d_18/conv1d_18", model.residual1d_block4.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_18/batch_normalization_18", model.residual1d_block4.bn2)

In [ ]:
# 5
assign_conv_weights(h5_file, "model_weights/conv1d_19/conv1d_19", model.residual1d_block5.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_19/batch_normalization_19", model.residual1d_block5.bn1)

assign_conv_weights(h5_file, "model_weights/conv1d_20/conv1d_20", model.residual1d_block5.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_20/batch_normalization_20", model.residual1d_block5.bn2)

In [ ]:
# 6
assign_conv_weights(h5_file, "model_weights/conv1d_21/conv1d_21", model.residual1d_block6.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_21/batch_normalization_21", model.residual1d_block6.bn1)

assign_conv_weights(h5_file, "model_weights/conv1d_22/conv1d_22", model.residual1d_block6.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_22/batch_normalization_22", model.residual1d_block6.bn2)

In [ ]:
# 7
assign_conv_weights(h5_file, "model_weights/conv1d_23/conv1d_23", model.residual1d_block7.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_23/batch_normalization_23", model.residual1d_block7.bn1)

assign_conv_weights(h5_file, "model_weights/conv1d_24/conv1d_24", model.residual1d_block7.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_24/batch_normalization_24", model.residual1d_block7.bn2)

In [ ]:
# 8
assign_conv_weights(h5_file, "model_weights/conv1d_25/conv1d_25", model.residual1d_block8.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_25/batch_normalization_25", model.residual1d_block8.bn1)

assign_conv_weights(h5_file, "model_weights/conv1d_26/conv1d_26", model.residual1d_block8.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_26/batch_normalization_26", model.residual1d_block8.bn2)

In [ ]:
# ConvBlockReduce
assign_conv_weights(h5_file, "model_weights/conv1d_27/conv1d_27", model.conv_reduce.layers[1])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_27/batch_normalization_27", model.conv_reduce.layers[2])

In [ ]:
def assign_conv2d_weights(h5_file, tf_layer_path, pytorch_conv2d_layer):
    """
    Assign weights from a TensorFlow Conv2d layer to a PyTorch Conv2d layer.
    
    Parameters:
        h5_file (h5py.File): The HDF5 file containing TensorFlow weights.
        tf_layer_path (str): Path to the TensorFlow Conv2d layer in the HDF5 file.
        pytorch_conv2d_layer (torch.nn.Conv2d): The PyTorch Conv2d layer.
    """
    # Access TensorFlow Conv2D weights
    tf_weights = h5_file[tf_layer_path]['kernel:0'][:]
    
    # Convert to PyTorch tensor
    pytorch_weights = torch.tensor(tf_weights, dtype=torch.float32)
    
    # Transpose TensorFlow weights to match PyTorch's format
    pytorch_weights = pytorch_weights.permute(3, 2, 0, 1)  # [output_channels, input_channels, filter_height, filter_width]
    
    # Ensure the shapes match
    assert pytorch_weights.shape == pytorch_conv2d_layer.weight.data.shape, \
        f"Shape mismatch: {pytorch_weights.shape} vs {pytorch_conv2d_layer.weight.data.shape}"
    
    # Assign weights to the PyTorch layer
    pytorch_conv2d_layer.weight.data = pytorch_weights

    print(f"Assigned Conv2D weights to PyTorch layer: {pytorch_conv2d_layer}")

In [ ]:
# HEAD

assign_conv2d_weights(h5_file, "model_weights/conv2d/conv2d", model.conv2d_block.block[1])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_28/batch_normalization_28", model.conv2d_block.block[2])

In [ ]:
# ResidualDilatedBlock - 2D

# 1
assign_conv2d_weights(h5_file, "model_weights/conv2d_1/conv2d_1", model.residual2d_block1.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_29/batch_normalization_29", model.residual2d_block1.bn1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_2/conv2d_2", model.residual2d_block1.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_30/batch_normalization_30", model.residual2d_block1.bn2)

In [ ]:
# 2
assign_conv2d_weights(h5_file, "model_weights/conv2d_3/conv2d_3", model.residual2d_block2.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_31/batch_normalization_31", model.residual2d_block2.bn1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_4/conv2d_4", model.residual2d_block2.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_32/batch_normalization_32", model.residual2d_block2.bn2)

In [ ]:
# 3
assign_conv2d_weights(h5_file, "model_weights/conv2d_5/conv2d_5", model.residual2d_block3.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_33/batch_normalization_33", model.residual2d_block3.bn1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_6/conv2d_6", model.residual2d_block3.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_34/batch_normalization_34", model.residual2d_block3.bn2)

In [ ]:
# 4
assign_conv2d_weights(h5_file, "model_weights/conv2d_7/conv2d_7", model.residual2d_block4.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_35/batch_normalization_35", model.residual2d_block4.bn1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_8/conv2d_8", model.residual2d_block4.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_36/batch_normalization_36", model.residual2d_block4.bn2)

In [ ]:
# 5
assign_conv2d_weights(h5_file, "model_weights/conv2d_9/conv2d_9", model.residual2d_block5.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_37/batch_normalization_37", model.residual2d_block5.bn1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_10/conv2d_10", model.residual2d_block5.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_38/batch_normalization_38", model.residual2d_block5.bn2)

In [ ]:
# 6
assign_conv2d_weights(h5_file, "model_weights/conv2d_11/conv2d_11", model.residual2d_block6.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_39/batch_normalization_39", model.residual2d_block6.bn1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_12/conv2d_12", model.residual2d_block6.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_40/batch_normalization_40", model.residual2d_block6.bn2)

In [ ]:
def assign_dense_weights(h5_file, tf_layer_path, pytorch_dense_layer):
    """
    Assign weights and biases from a TensorFlow Dense layer to a PyTorch Dense layer.
    
    Parameters:
        h5_file (h5py.File): The HDF5 file containing TensorFlow weights.
        tf_layer_path (str): Path to the TensorFlow dense layer in the HDF5 file.
        pytorch_dense_layer (torch.nn.Linear): The PyTorch dense (fully connected) layer.
    """
    # Access TensorFlow Dense layer parameters (weights and biases)
    tf_kernel = h5_file[tf_layer_path]['kernel:0'][:]  # Weights (input_units, output_units)
    tf_bias = h5_file[tf_layer_path]['bias:0'][:]      # Bias (output_units,)
    
    # Convert to PyTorch tensors
    pytorch_weights = torch.tensor(tf_kernel, dtype=torch.float32)
    pytorch_bias = torch.tensor(tf_bias, dtype=torch.float32)
    
    # Transpose TensorFlow weights to match PyTorch's format
    pytorch_weights = pytorch_weights.t()  # (output_units, input_units)
    
    # Ensure the shapes match
    assert pytorch_weights.shape == pytorch_dense_layer.weight.data.shape, \
        f"Shape mismatch: {pytorch_weights.shape} vs {pytorch_dense_layer.weight.data.shape}"
    assert pytorch_bias.shape == pytorch_dense_layer.bias.data.shape, \
        f"Shape mismatch: {pytorch_bias.shape} vs {pytorch_dense_layer.bias.data.shape}"
    
    # Assign weights and biases to the PyTorch layer
    pytorch_dense_layer.weight.data = pytorch_weights
    pytorch_dense_layer.bias.data = pytorch_bias

    print(f"Assigned Dense layer weights and biases to PyTorch layer: {pytorch_dense_layer}")


In [ ]:
# Final
assign_dense_weights(h5_file, "model_weights/dense/dense", model.final.dense)

In [ ]:
# Save the entire model
torch.save(model, "model.pth")